In [2]:
import pandas as pd
from os import listdir
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
from random import randint
import numpy as np
import math
from tensorflow import keras

In [4]:
# Data reading/preprocessing
data = pd.DataFrame()

# This determines which minutes the model gets access to
# Here the model is allowed to see prices from 10-60 minutes ago
steps = list(range(15,61))
step_col = lambda step: f"back_{step}"
last_col = step_col(min(steps))

frames = 0
for ticker in listdir("data"):
    all_days = []
    for day in listdir(f"data/{ticker}"):
        all_days.append(pd.read_parquet(f"data/{ticker}/{day}"))
    all_prices = pd.concat(all_days)[["open"]]

    if frames > 30:
        break

    # Normalize linearly to 0=0, 1=mean
    # This preserves ratios and so preserves percentage increase/decrease
    # mean = all_prices["open"].mean()

    for day_data in all_days:
        # actually let's do the means per day, since obviously prices change over time
        mean = day_data["open"].mean()
        # Also taking the logarithm of the normalized data, since it makes ratios into differences
        day_data["open"] = np.log(day_data["open"] / mean)
        for step in steps:
            day_data[step_col(step)] = day_data["open"].shift(step)
        day_data = day_data.iloc[max(steps):] # clear out NaNs created by steps
        # new_data = day_data[["open"]]
        # for (i, step) in enumerate(steps):
        #     if i == 0:
        #         continue
        #     new_data[step_col(step)] = day_data[step_col(step)] / day_data[step_col(steps[i-1])]
        data = pd.concat([data, day_data], ignore_index=True)
        frames += 1

print(f"{len(data)} entries from {frames} days")

37193 entries from 58 days


In [ ]:
X = data.drop(columns=["open", "time"])
y = data["open"] - data[last_col]

X = np.array(X)[..., np.newaxis]
y = np.array(y)

train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.15, random_state=1)
train_X, val_X, train_y, val_y = train_test_split(train_X, train_y, test_size=0.15, random_state=1)

In [6]:
test_X.shape

(5579, 46, 1)

In [4]:
model = keras.Sequential([
    keras.layers.Input(shape=(10, 1)),
    keras.layers.LSTM(20),
    keras.layers.Dense(1)
])

model.compile(optimizer="adam", loss="mse")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 20)             │         1,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,781 (6.96 KB)

 Trainable params: 1,781 (6.96 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True)

model.fit(train_X, train_y, validation_data=(
    val_X, val_y), epochs=10, batch_size=32, callbacks=[early_stopping])

Epoch 1/10
 8606/16918 ━━━━━━━━━━━━━━━━━━━━ 2:44 20ms/step - loss: 2.2963e-05

KeyboardInterrupt: 

In [ ]:
root_mean_squared_error(test_y, model.predict(test_X))

1281/1281 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step


0.0011996963866520748

In [ ]:
pred_y = np.exp(model.predict(test_X)).transpose()[0].tolist()
true_y = np.exp(test_y).tolist()

1281/1281 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step


In [1]:
# one-minute predictions and yet it can't seem to do much better than a random guess
idx = randint(0, len(pred_y))

pairs = [(val, -step) for (val, step) in zip(test_X[idx].transpose()[0].tolist(), reversed(steps))]
pairs.sort(key=lambda p: p[1])

x = [t for (_, t) in pairs]
y = [math.exp(v) for (v, _) in pairs]

(last_y, last_x) = pairs[-1]

plt.figure()
plt.plot(x, y)
plt.plot([0, last_x], [pred_y[idx], math.exp(last_y)], 'r--')
plt.plot([0, last_x], [true_y[idx], math.exp(last_y)], 'b--')
plt.show()

NameError: name 'randint' is not defined